
# STEP 34B — Publication Table 1 Generator
## BrainFMOps-Analyze

**Input**

```text
<repository-root>\
└── 34A_Cohort_Merger_Output\
    └── Merged_Cohort_212.xlsx
```

**Purpose**

สร้าง Table 1 สำหรับบทความจาก cohort ที่ผ่านการตรวจสอบแล้ว 212 subjects:

- CN = 124
- AD = 88
- Matched clinical records = 212
- Unmatched = 0

Notebook นี้จะ:

1. ตรวจ cohort ซ้ำก่อนคำนวณ
2. ตรวจจับตัวแปร OASIS อัตโนมัติ
3. คำนวณ descriptive statistics
4. คำนวณ Welch's t-test สำหรับตัวแปรต่อเนื่อง
5. คำนวณ Chi-square/Fisher's exact test สำหรับเพศ
6. รายงาน CDR แบบ descriptive เพราะ CDR มีความสัมพันธ์กับนิยามกลุ่ม
7. สร้าง Missing-data และ Statistical Assumption reports
8. Export Table 1 เป็น Excel/CSV
9. สร้าง Word table แบบ optional ถ้ามี `python-docx`

> ใช้งาน: **Kernel → Restart & Run All**


In [ ]:

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

ROOT_CANDIDATES = [
    Path.cwd().resolve(),
    Path.cwd().resolve(),
    Path.cwd(),
]

ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), Path.cwd())
INPUT_DIR = ROOT / "34A_Cohort_Merger_Output"
OUTPUT_DIR = ROOT / "34B_Table1_Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = INPUT_DIR / "Merged_Cohort_212.xlsx"

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"ไม่พบไฟล์ {INPUT_FILE}\n"
        "กรุณารัน STEP 34A ให้ผ่านก่อน"
    )

print("INPUT :", INPUT_FILE)
print("OUTPUT:", OUTPUT_DIR)


In [ ]:

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def first_existing(columns, candidates):
    lookup = {str(c).strip().lower(): c for c in columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    return None

def numeric_series(df, col):
    if col is None:
        return pd.Series(dtype=float)
    return pd.to_numeric(df[col], errors="coerce")

def fmt_mean_sd(series, digits=2):
    x = pd.to_numeric(series, errors="coerce").dropna()
    if len(x) == 0:
        return "NA"
    return f"{x.mean():.{digits}f} ± {x.std(ddof=1):.{digits}f}"

def fmt_median_iqr(series, digits=1):
    x = pd.to_numeric(series, errors="coerce").dropna()
    if len(x) == 0:
        return "NA"
    q1, med, q3 = np.percentile(x, [25, 50, 75])
    return f"{med:.{digits}f} ({q1:.{digits}f}–{q3:.{digits}f})"

def fmt_n_pct(n, total):
    if total == 0:
        return "0 (0.0%)"
    return f"{int(n)} ({100*n/total:.1f}%)"

def fmt_p(p):
    if p is None or not np.isfinite(p):
        return "—"
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"

def welch_test(a, b):
    a = pd.to_numeric(a, errors="coerce").dropna()
    b = pd.to_numeric(b, errors="coerce").dropna()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    return float(stats.ttest_ind(a, b, equal_var=False).pvalue)

def mann_whitney(a, b):
    a = pd.to_numeric(a, errors="coerce").dropna()
    b = pd.to_numeric(b, errors="coerce").dropna()
    if len(a) == 0 or len(b) == 0:
        return np.nan
    return float(stats.mannwhitneyu(a, b, alternative="two-sided").pvalue)

def shapiro_p(series):
    x = pd.to_numeric(series, errors="coerce").dropna()
    if len(x) < 3:
        return np.nan
    # scipy Shapiro is intended for <=5000 for p-value accuracy
    if len(x) > 5000:
        x = x.sample(5000, random_state=42)
    return float(stats.shapiro(x).pvalue)

def levene_p(a, b):
    a = pd.to_numeric(a, errors="coerce").dropna()
    b = pd.to_numeric(b, errors="coerce").dropna()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    return float(stats.levene(a, b, center="median").pvalue)

def hedges_g(a, b):
    a = pd.to_numeric(a, errors="coerce").dropna().to_numpy()
    b = pd.to_numeric(b, errors="coerce").dropna().to_numpy()
    n1, n2 = len(a), len(b)
    if n1 < 2 or n2 < 2:
        return np.nan
    s1, s2 = np.std(a, ddof=1), np.std(b, ddof=1)
    pooled = math.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2)/(n1+n2-2))
    if pooled == 0:
        return 0.0
    d = (np.mean(a) - np.mean(b)) / pooled
    correction = 1 - 3 / (4*(n1+n2) - 9)
    return float(correction * d)

def sex_normalize(v):
    if pd.isna(v):
        return np.nan
    t = str(v).strip().upper()
    mapping = {
        "F": "F", "FEMALE": "F", "WOMAN": "F",
        "M": "M", "MALE": "M", "MAN": "M",
    }
    return mapping.get(t, np.nan)

def chi_or_fisher(table):
    arr = np.asarray(table, dtype=int)
    if arr.shape != (2, 2) or arr.sum() == 0:
        return np.nan, "NA"
    chi2, p, dof, expected = stats.chi2_contingency(arr)
    if (expected < 5).any():
        _, p = stats.fisher_exact(arr)
        return float(p), "Fisher's exact test"
    return float(p), "Chi-square test"


In [ ]:

# ------------------------------------------------------------
# Load merged cohort
# ------------------------------------------------------------
df = pd.read_excel(INPUT_FILE)

print("Shape:", df.shape)
print("Columns:")
print(list(df.columns))

GROUP_CANDIDATES = ["_group", "group", "clinical_label", "ground_truth"]
group_col = first_existing(df.columns, GROUP_CANDIDATES)

if group_col is None:
    raise KeyError("ไม่พบ group column (CN/AD)")

df["_group_pub"] = df[group_col].astype(str).str.strip().str.upper()
df = df[df["_group_pub"].isin(["CN", "AD"])].copy()

cn = df[df["_group_pub"] == "CN"].copy()
ad = df[df["_group_pub"] == "AD"].copy()

print("\nCohort:")
print("CN   :", len(cn))
print("AD   :", len(ad))
print("Total:", len(df))

if len(df) != 212 or len(cn) != 124 or len(ad) != 88:
    raise ValueError(
        "Cohort ไม่ตรงกับ STEP 34A: "
        f"CN={len(cn)}, AD={len(ad)}, Total={len(df)}"
    )


In [ ]:

# ------------------------------------------------------------
# Detect OASIS clinical variables
# ------------------------------------------------------------
CANDIDATES = {
    "sex": ["M/F", "sex", "gender"],
    "age": ["Age", "age_years"],
    "education": ["Educ", "education", "education_years"],
    "ses": ["SES", "socioeconomic_status"],
    "mmse": ["MMSE", "mmse_score"],
    "cdr": ["CDR", "cdr_score"],
    "etiv": ["eTIV", "etiv", "estimated_total_intracranial_volume"],
    "nwbv": ["nWBV", "nwbv", "normalized_whole_brain_volume"],
    "asf": ["ASF", "atlas_scaling_factor"],
    "slice_count": [
        "n_slices", "slice_count", "slices_analysed", "slices_analyzed",
        "selected_slice_count", "num_slices", "number_of_slices"
    ],
}

cols = {
    key: first_existing(df.columns, values)
    for key, values in CANDIDATES.items()
}

print("Detected variables:")
for k, v in cols.items():
    print(f"{k:12s}: {v}")


In [ ]:

# ------------------------------------------------------------
# Data-quality and missingness audit
# ------------------------------------------------------------
audit_rows = []

for key, col in cols.items():
    if col is None:
        audit_rows.append({
            "Variable": key,
            "Detected column": "NOT FOUND",
            "Non-missing n": 0,
            "Missing n": len(df),
            "Missing %": 100.0,
        })
        continue

    nonmissing = int(df[col].notna().sum())
    missing = int(df[col].isna().sum())

    audit_rows.append({
        "Variable": key,
        "Detected column": col,
        "Non-missing n": nonmissing,
        "Missing n": missing,
        "Missing %": round(100 * missing / len(df), 2),
    })

missing_report = pd.DataFrame(audit_rows)
display(missing_report)


In [ ]:

# ------------------------------------------------------------
# Continuous-variable statistical report
# Primary test in Table 1 = Welch's t-test
# Mann-Whitney is provided as sensitivity analysis.
# ------------------------------------------------------------
continuous_specs = [
    ("age", "Age (years)"),
    ("education", "Education (years)"),
    ("mmse", "MMSE"),
    ("etiv", "eTIV"),
    ("nwbv", "nWBV"),
]

stats_rows = []

for key, label in continuous_specs:
    col = cols.get(key)
    if col is None:
        continue

    cn_x = numeric_series(cn, col)
    ad_x = numeric_series(ad, col)

    stats_rows.append({
        "Variable": label,
        "CN n": int(cn_x.notna().sum()),
        "AD n": int(ad_x.notna().sum()),
        "CN mean": cn_x.mean(),
        "CN SD": cn_x.std(ddof=1),
        "AD mean": ad_x.mean(),
        "AD SD": ad_x.std(ddof=1),
        "CN Shapiro p": shapiro_p(cn_x),
        "AD Shapiro p": shapiro_p(ad_x),
        "Levene p": levene_p(cn_x, ad_x),
        "Welch p": welch_test(cn_x, ad_x),
        "Mann-Whitney p": mann_whitney(cn_x, ad_x),
        "Hedges g (CN-AD)": hedges_g(cn_x, ad_x),
    })

statistics_report = pd.DataFrame(stats_rows)
display(statistics_report)


In [ ]:

# ------------------------------------------------------------
# Build publication Table 1
# ------------------------------------------------------------
table_rows = []

# Age
if cols["age"] is not None:
    p = welch_test(numeric_series(cn, cols["age"]), numeric_series(ad, cols["age"]))
    table_rows.append([
        "Age (years), mean ± SD",
        fmt_mean_sd(cn[cols["age"]], 1),
        fmt_mean_sd(ad[cols["age"]], 1),
        fmt_mean_sd(df[cols["age"]], 1),
        fmt_p(p),
    ])

# Sex
if cols["sex"] is not None:
    sex_all = df[cols["sex"]].map(sex_normalize)
    sex_cn = cn[cols["sex"]].map(sex_normalize)
    sex_ad = ad[cols["sex"]].map(sex_normalize)

    cn_f, cn_m = int((sex_cn == "F").sum()), int((sex_cn == "M").sum())
    ad_f, ad_m = int((sex_ad == "F").sum()), int((sex_ad == "M").sum())
    all_f, all_m = int((sex_all == "F").sum()), int((sex_all == "M").sum())

    p_sex, sex_test = chi_or_fisher([[cn_f, cn_m], [ad_f, ad_m]])

    table_rows.append([
        "Sex, female / male, n (%)",
        f"{fmt_n_pct(cn_f, len(cn))} / {fmt_n_pct(cn_m, len(cn))}",
        f"{fmt_n_pct(ad_f, len(ad))} / {fmt_n_pct(ad_m, len(ad))}",
        f"{fmt_n_pct(all_f, len(df))} / {fmt_n_pct(all_m, len(df))}",
        fmt_p(p_sex),
    ])
else:
    sex_test = "NA"

# Education, MMSE, eTIV, nWBV
for key, label, digits in [
    ("education", "Education (years), mean ± SD", 1),
    ("mmse", "MMSE, mean ± SD", 1),
    ("etiv", "eTIV, mean ± SD", 1),
    ("nwbv", "nWBV, mean ± SD", 3),
]:
    col = cols.get(key)
    if col is None:
        continue

    p = welch_test(numeric_series(cn, col), numeric_series(ad, col))
    table_rows.append([
        label,
        fmt_mean_sd(cn[col], digits),
        fmt_mean_sd(ad[col], digits),
        fmt_mean_sd(df[col], digits),
        fmt_p(p),
    ])

# CDR distribution — descriptive only
if cols["cdr"] is not None:
    cdr_col = cols["cdr"]

    def cdr_counts(frame, selector):
        x = pd.to_numeric(frame[cdr_col], errors="coerce")
        return int(selector(x).sum())

    cdr_categories = [
        ("CDR 0, n (%)", lambda x: x == 0),
        ("CDR 0.5, n (%)", lambda x: x == 0.5),
        ("CDR 1, n (%)", lambda x: x == 1),
        ("CDR ≥ 2, n (%)", lambda x: x >= 2),
    ]

    for label, selector in cdr_categories:
        cn_n = cdr_counts(cn, selector)
        ad_n = cdr_counts(ad, selector)
        all_n = cdr_counts(df, selector)

        table_rows.append([
            label,
            fmt_n_pct(cn_n, len(cn)),
            fmt_n_pct(ad_n, len(ad)),
            fmt_n_pct(all_n, len(df)),
            "—",
        ])

# Slice count, only if actual data exists
if cols["slice_count"] is not None and numeric_series(df, cols["slice_count"]).notna().any():
    p = mann_whitney(
        numeric_series(cn, cols["slice_count"]),
        numeric_series(ad, cols["slice_count"])
    )
    table_rows.append([
        "MRI slices analysed per subject, median (IQR)",
        fmt_median_iqr(cn[cols["slice_count"]], 0),
        fmt_median_iqr(ad[cols["slice_count"]], 0),
        fmt_median_iqr(df[cols["slice_count"]], 0),
        fmt_p(p),
    ])

table1 = pd.DataFrame(
    table_rows,
    columns=[
        "Characteristic",
        "CN (n = 124)",
        "AD (n = 88)",
        "All (n = 212)",
        "p-value",
    ],
)

display(table1)


In [ ]:

# ------------------------------------------------------------
# Publication footnote + reviewer notes
# ------------------------------------------------------------
footnote = (
    "CN = cognitively normal; AD = Alzheimer's disease; "
    "SD = standard deviation; MMSE = Mini-Mental State Examination; "
    "CDR = Clinical Dementia Rating; eTIV = estimated total intracranial volume; "
    "nWBV = normalized whole-brain volume. "
    "Continuous-variable p-values were computed using Welch's independent-samples "
    "t-test. Sex was compared using "
    + sex_test
    + ". CDR was reported descriptively because it contributes to the clinical "
      "group definition. Missing values were excluded pairwise from each analysis."
)

reviewer_notes = []

if cols["slice_count"] is None:
    reviewer_notes.append(
        "MRI slice-count variable was not found in Merged_Cohort_212.xlsx; "
        "the row was omitted from the publication table."
    )

for key in ["age", "sex", "education", "mmse", "cdr", "etiv", "nwbv"]:
    col = cols.get(key)
    if col is None:
        reviewer_notes.append(f"Required variable not found: {key}")
    elif df[col].isna().any():
        reviewer_notes.append(
            f"{key}: {int(df[col].isna().sum())} missing values "
            f"({100*df[col].isna().mean():.1f}%)."
        )

print("FOOTNOTE:\n", footnote)
print("\nREVIEWER NOTES:")
for note in reviewer_notes:
    print("-", note)


In [ ]:

# ------------------------------------------------------------
# Plausibility audit
# This flags suspicious values; it does NOT alter the dataset.
# ------------------------------------------------------------
plausibility_rules = {
    "age": (18, 110),
    "education": (0, 30),
    "mmse": (0, 30),
    "cdr": (0, 5),
    "etiv": (500, 3000),
    "nwbv": (0, 1.2),
}

plausibility_rows = []

for key, (lo, hi) in plausibility_rules.items():
    col = cols.get(key)
    if col is None:
        continue

    x = pd.to_numeric(df[col], errors="coerce")
    bad = x.notna() & ((x < lo) | (x > hi))

    plausibility_rows.append({
        "Variable": key,
        "Range checked": f"{lo} to {hi}",
        "Non-missing n": int(x.notna().sum()),
        "Out-of-range n": int(bad.sum()),
        "Min observed": x.min(),
        "Max observed": x.max(),
    })

plausibility_report = pd.DataFrame(plausibility_rows)
display(plausibility_report)


In [ ]:

# ------------------------------------------------------------
# Export CSV + XLSX
# ------------------------------------------------------------
csv_path = OUTPUT_DIR / "Table1_Demographic_Clinical_Characteristics.csv"
xlsx_path = OUTPUT_DIR / "Table1_Demographic_Clinical_Characteristics.xlsx"
stats_path = OUTPUT_DIR / "Table1_Statistics_Report.xlsx"

table1.to_csv(csv_path, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    table1.to_excel(writer, sheet_name="Table 1", index=False, startrow=2)

    ws = writer.book["Table 1"]
    ws["A1"] = (
        "Table 1. Demographic and clinical characteristics of the 212 subjects "
        "included in the quantitative subject-level evaluation."
    )

    footnote_row = len(table1) + 5
    ws.cell(row=footnote_row, column=1, value=footnote)

    # Basic publication formatting
    from openpyxl.styles import Font, Alignment, Border, Side
    thin = Side(style="thin", color="000000")

    ws["A1"].font = Font(bold=True, size=11)

    for cell in ws[3]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = Border(top=thin, bottom=thin)

    for row in ws.iter_rows(min_row=4, max_row=3+len(table1), min_col=1, max_col=5):
        for cell in row:
            cell.alignment = Alignment(vertical="center")
        for cell in row[1:]:
            cell.alignment = Alignment(horizontal="center", vertical="center")

    for col_letter, width in {
        "A": 42, "B": 24, "C": 24, "D": 24, "E": 13
    }.items():
        ws.column_dimensions[col_letter].width = width

    ws.cell(row=footnote_row, column=1).alignment = Alignment(
        wrap_text=True, vertical="top"
    )
    ws.merge_cells(
        start_row=footnote_row, start_column=1,
        end_row=footnote_row+2, end_column=5
    )

with pd.ExcelWriter(stats_path, engine="openpyxl") as writer:
    statistics_report.to_excel(
        writer, sheet_name="Continuous Statistics", index=False
    )
    missing_report.to_excel(
        writer, sheet_name="Missing Data", index=False
    )
    plausibility_report.to_excel(
        writer, sheet_name="Plausibility Audit", index=False
    )

print("Saved:")
print(csv_path)
print(xlsx_path)
print(stats_path)


In [ ]:

# ------------------------------------------------------------
# Optional Word table
# ------------------------------------------------------------
docx_path = OUTPUT_DIR / "Table1_Publication.docx"

try:
    from docx import Document
    from docx.shared import Pt
    from docx.enum.text import WD_ALIGN_PARAGRAPH

    document = Document()

    p = document.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = p.add_run(
        "Table 1. Demographic and clinical characteristics of the 212 subjects "
        "included in the quantitative subject-level evaluation."
    )
    run.bold = True
    run.font.size = Pt(10)

    table = document.add_table(
        rows=1,
        cols=len(table1.columns)
    )
    table.style = "Table Grid"

    for j, col in enumerate(table1.columns):
        table.rows[0].cells[j].text = str(col)

    for _, row in table1.iterrows():
        cells = table.add_row().cells
        for j, value in enumerate(row):
            cells[j].text = str(value)

    fp = document.add_paragraph()
    fr = fp.add_run(footnote)
    fr.italic = True
    fr.font.size = Pt(8)

    document.save(docx_path)
    print("Saved:", docx_path)

except Exception as e:
    print("Word export skipped:", e)


In [ ]:

# ------------------------------------------------------------
# Final audit and manifest
# ------------------------------------------------------------
critical = {
    "Total subjects = 212": len(df) == 212,
    "CN = 124": len(cn) == 124,
    "AD = 88": len(ad) == 88,
    "Age found": cols["age"] is not None,
    "Sex found": cols["sex"] is not None,
    "MMSE found": cols["mmse"] is not None,
    "CDR found": cols["cdr"] is not None,
    "eTIV found": cols["etiv"] is not None,
    "nWBV found": cols["nwbv"] is not None,
}

validation = pd.DataFrame(
    [{"Check": k, "Pass": bool(v)} for k, v in critical.items()]
)
display(validation)

manifest = {
    "input_file": str(INPUT_FILE),
    "output_directory": str(OUTPUT_DIR),
    "cohort": {
        "CN": int(len(cn)),
        "AD": int(len(ad)),
        "All": int(len(df)),
    },
    "detected_columns": cols,
    "continuous_test_primary": "Welch independent-samples t-test",
    "sex_test": sex_test,
    "cdr_test": "Descriptive only",
    "footnote": footnote,
    "reviewer_notes": reviewer_notes,
    "all_critical_checks_passed": bool(validation["Pass"].all()),
}

with open(
    OUTPUT_DIR / "Table1_Audit_Report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

with open(
    OUTPUT_DIR / "Table1_Log.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write("BrainFMOps-Analyze STEP 34B\n")
    f.write("="*60 + "\n")
    f.write(f"CN: {len(cn)}\nAD: {len(ad)}\nTotal: {len(df)}\n")
    f.write(f"Sex test: {sex_test}\n")
    f.write("\nReviewer notes:\n")
    for note in reviewer_notes:
        f.write(f"- {note}\n")

if validation["Pass"].all():
    print("\nTABLE 1 CORE VARIABLES READY FOR PUBLICATION")
else:
    print("\nATTENTION: Some required variables were not detected.")



## Output files

```text
34B_Table1_Output
├── Table1_Demographic_Clinical_Characteristics.xlsx
├── Table1_Demographic_Clinical_Characteristics.csv
├── Table1_Publication.docx
├── Table1_Statistics_Report.xlsx
├── Table1_Audit_Report.json
└── Table1_Log.txt
```

### สิ่งที่ต้องตรวจหลัง Run

1. Header ต้องเป็น `CN (n = 124)`, `AD (n = 88)`, `All (n = 212)`
2. Age, Sex, MMSE, CDR, eTIV, nWBV ต้องมีค่าจริง ไม่ใช่ `NA`
3. Education อาจมี missing values ตาม OASIS และต้องรายงานตามจริง
4. CDR ไม่มี p-value เพราะเกี่ยวข้องกับนิยาม clinical group
5. ถ้า slice-count ไม่พบ Notebook จะ **ตัดแถวนั้นออก** แทนการใส่ค่าปลอม
6. ตรวจ `Table1_Statistics_Report.xlsx` ก่อน copy Table 1 ลง manuscript
